<a href="https://colab.research.google.com/github/ikbal66123/biopay/blob/claude%2Fbold-allen-vmzjvm/anti-deepfake/colab/anti_deepfake_1klik.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🛡️ Anti-Deepfake — versi 1 klik

**Cara pakai (3 langkah):**
1. Menu **Runtime → Run all** (atau tekan `Ctrl+F9`). Kalau muncul peringatan, klik **Run anyway**.
2. Tunggu ±5–10 menit sampai muncul kotak biru **"Aplikasi siap! Buka: https://…"**.
3. Klik link itu (bisa juga dibuka di HP) → upload foto/video → **Download hasil**.

Tidak perlu akun GitHub atau token: seluruh kode aplikasi sudah ada di dalam notebook ini.
Kalau muncul tulisan *GPU belum aktif*: menu **Runtime → Change runtime type → T4 GPU → Save**, lalu ulangi langkah 1.


In [4]:
#@title (kode aplikasi tertanam — tidak perlu dibuka) { display-mode: "form" }
CODE_VERSION = 'd7276c9'
CODE = '...'


In [5]:
#@title ▶ Klik tombol play di kiri ini (atau Runtime → Run all), lalu tunggu link muncul { display-mode: "form" }
PRESET = "balanced"  #@param ["fast", "balanced", "strong"]
import base64, io, os, re, subprocess, sys, tarfile, time, urllib.request
from IPython.display import HTML, display, clear_output

def say(msg):
    print(msg, flush=True)

def sh(cmd):
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if r.returncode != 0:
        print(r.stdout[-2000:], r.stderr[-2000:])
        raise SystemExit(f"❌ Perintah gagal: {cmd}")
    return r.stdout

# 1) GPU
import torch
if torch.cuda.is_available():
    say(f"✅ 1/5 GPU aktif: {torch.cuda.get_device_name(0)}")
    DEVICE = "cuda"
else:
    DEVICE = "cpu"
    display(HTML("<h3 style='color:#c62828'>⚠️ GPU belum aktif — aplikasi tetap jalan tapi SANGAT lambat.</h3>"
                 "<p>Untuk GPU: menu <b>Runtime → Change runtime type → T4 GPU → Save</b>, lalu klik play lagi.</p>"))

# 2) Kode aplikasi (tertanam di notebook ini, tidak perlu GitHub)
APP = "/content/anti-deepfake"
if (open(f"{APP}/.version").read().strip() if os.path.exists(f"{APP}/.version") else "") != CODE_VERSION:
    subprocess.run(f"rm -rf {APP}", shell=True)  # kode lama dari sesi sebelumnya -> ganti
    with tarfile.open(fileobj=io.BytesIO(base64.b64decode(CODE)), mode="r:gz") as t:
        t.extractall("/content", filter="data")
    open(f"{APP}/.version", "w").write(CODE_VERSION)
os.chdir(APP)
say(f"✅ 2/5 Kode aplikasi siap (versi {CODE_VERSION})")

# 3) Library (PyTorch+CUDA sudah ada di Colab)
if not os.path.exists("/content/.adf_installed"):
    say("⏳ 3/5 Memasang library (±3–6 menit, sekali per sesi)…")
    sh(f"{sys.executable} -m pip install -q diffusers transformers accelerate scikit-image tqdm "
       "insightface onnxruntime-gpu onnx2torch open_clip_torch fastapi uvicorn python-multipart")
    sh(f"{sys.executable} -m pip install -q --no-deps invisible-watermark")
    ok = subprocess.run([sys.executable, "-c", "import cv2; assert hasattr(cv2, 'CascadeClassifier')"]).returncode == 0
    if not ok:  # insightface kadang menarik OpenCV 5 (tanpa Haar cascade)
        subprocess.run(f"{sys.executable} -m pip uninstall -y -q opencv-python opencv-python-headless opencv-contrib-python", shell=True)
        sh(f"{sys.executable} -m pip install -q 'opencv-python-headless<5'")
    if subprocess.run("which ffmpeg", shell=True).returncode != 0:
        sh("apt-get -qq install -y ffmpeg")
    open("/content/.adf_installed", "w").close()
say("✅ 3/5 Library terpasang")

# 4) Nyalakan aplikasi (model diunduh otomatis saat pertama dipakai)
PORT = 8000
subprocess.run("pkill -f 'app/server.py'; pkill -f cloudflared", shell=True)
time.sleep(1)
say("⏳ 4/5 Mengunduh model AI (VAE SD1.5+SDXL, CLIP ViT-L/14, SigLIP, ArcFace) & menyalakan aplikasi (±3–6 menit)…")
warm = subprocess.run([sys.executable, "-c",
    "import sys; sys.path.insert(0,'.');"
    "from antideepfake.pipeline import get_vaes, get_embedders, ENSEMBLE_VAES, ENSEMBLE_EMBEDDERS;"
    "import torch; d=torch.device('cpu'); get_vaes(ENSEMBLE_VAES, d, strict=False); get_embedders(ENSEMBLE_EMBEDDERS, d, strict=False);"
    "from antideepfake.faces import FaceDetector; FaceDetector('insightface');"
    "from antideepfake.identity_attack import ArcFaceTorch; ArcFaceTorch('cpu')"], capture_output=True, text=True)
if warm.returncode != 0:
    print(warm.stderr[-3000:]); raise SystemExit("❌ Gagal mengunduh model (lihat pesan di atas)")
env = dict(os.environ, ANTIDEEPFAKE_DEVICE=DEVICE, ANTIDEEPFAKE_PRESET=PRESET,
           ANTIDEEPFAKE_DATA="/content/app_data", HOST="127.0.0.1", PORT=str(PORT))
server = subprocess.Popen([sys.executable, "app/server.py"], env=env,
                          stdout=open("/content/server.log", "w"), stderr=subprocess.STDOUT)
for _ in range(90):
    try:
        urllib.request.urlopen(f"http://127.0.0.1:{PORT}/api/config", timeout=2); break
    except Exception:
        if server.poll() is not None:
            print(open("/content/server.log").read()[-3000:]); raise SystemExit("❌ Aplikasi gagal menyala")
        time.sleep(1)
say("✅ 4/5 Aplikasi menyala")

# 5) Link publik (Cloudflare Quick Tunnel, gratis tanpa akun)
if not os.path.exists("/content/cloudflared"):
    urllib.request.urlretrieve("https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64",
                               "/content/cloudflared")
    os.chmod("/content/cloudflared", 0o755)
subprocess.Popen(["/content/cloudflared", "tunnel", "--no-autoupdate", "--url", f"http://127.0.0.1:{PORT}"],
                 stdout=open("/content/tunnel.log", "w"), stderr=subprocess.STDOUT)
url = None
for _ in range(60):
    m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", open("/content/tunnel.log").read())
    if m:
        url = m.group(0); break
    time.sleep(1)
backup = None
try:
    from google.colab.output import eval_js
    backup = eval_js(f"google.colab.kernel.proxyPort({PORT})")
except Exception:
    pass
time.sleep(4)
html = "<div style='font-family:system-ui;padding:16px;border:2px solid #2f6fed;border-radius:12px'>"
if url:
    html += (f"<h2>✅ 5/5 Aplikasi siap! Buka: <a href='{url}' target='_blank'>{url}</a></h2>"
             "<p>Bisa dibuka dari HP juga. Kalau muncul error, tunggu 20 detik lalu refresh.</p>")
if backup:
    html += f"<p>Link cadangan (hanya di browser yang login akun Google ini): <a href='{backup}' target='_blank'>{backup}</a></p>"
html += ("<p>⚠️ Biarkan tab Colab ini tetap terbuka. Jangan sebar link-nya: siapa pun yang punya link bisa memakai "
         "aplikasimu. Semua file hilang saat sesi Colab berakhir.</p></div>")
display(HTML(html))


✅ 1/5 GPU aktif: Tesla T4
✅ 2/5 Kode aplikasi siap (versi d7276c9)
✅ 3/5 Library terpasang
⏳ 4/5 Mengunduh model AI (VAE SD1.5+SDXL, CLIP ViT-L/14, SigLIP, ArcFace) & menyalakan aplikasi (±3–6 menit)…
✅ 4/5 Aplikasi menyala


In [6]:
#@title (Kalau ada masalah) lihat log { display-mode: "form" }
for f in ['/content/server.log', '/content/tunnel.log']:
    print('=====', f); print(''.join(open(f).readlines()[-30:]) if os.path.exists(f) else '(belum ada)')


===== /content/server.log
INFO:     Started server process [48169]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://127.0.0.1:8000 (Press CTRL+C to quit)
INFO:     127.0.0.1:48708 - "GET /api/config HTTP/1.1" 200 OK

===== /content/tunnel.log
2026-11-06T11:11:17Z INF Thank you for trying Cloudflare Tunnel. Doing so, without a Cloudflare account, is a quick way to experiment and try it out. However, be aware that these account-less Tunnels have no uptime guarantee, are subject to the Cloudflare Online Services Terms of Use (https://www.cloudflare.com/website-terms/), and Cloudflare reserves the right to investigate your use of Tunnels for violations of such terms. If you intend to use Tunnels in production you should use a pre-created named tunnel by following: https://developers.cloudflare.com/cloudflare-one/connections/connect-apps
2026-11-06T11:11:17Z INF Requesting new quick Tunnel on trycloudflare.com...
2026-11-